In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import numpy as np
from scipy.stats import gmean

In [17]:
# Read in the cd8_t expression results
cd8_t = pd.read_csv('../data/all_cd8_t_cells.csv')


In [18]:
cd8_t.head()

,Unnamed: 0,209Bi_CD45,Center,164Dy,166Er_CD34,Event_length,157Gd,113In_CD45,191Ir_DNA1,193Ir_DNA2,...,152Sm_CD159c_asinh_coarseAlign_fineAlign,169Tm_CD25_asinh_coarseAlign_fineAlign,171Yb_Granzyme_B_asinh_coarseAlign_fineAlign,172Yb_CD38_asinh_coarseAlign_fineAlign,173Yb_CD14_asinh_coarseAlign_fineAlign,174Yb_HLA-DR_asinh_coarseAlign_fineAlign,176Yb_CD56_asinh_coarseAlign_fineAlign,Alignment_MC_fineAlign,FlowSOM_cluster,FlowSOM_metacluster
0,1,0.000000,703.693,0.0,0.00000,17,0.000000,0.0,218.307,384.787,...,0.056854,0.011151,5.832982,2.280988,0.01453,0.647175,0.054384,7,70,22
1,2,0.050212,791.971,0.0,0.00000,20,0.000000,0.0,156.895,337.880,...,0.006105,0.011151,3.614947,1.015876,0.01453,0.073580,0.054384,7,90,6
2,3,0.000000,0.000,0.0,7.96826,14,0.000000,0.0,139.201,343.646,...,0.006105,0.011151,2.196237,2.859861,0.56924,0.437788,0.054384,7,194,39
3,4,0.000000,794.320,0.0,0.00000,17,0.927911,0.0,149.295,344.580,...,0.006105,0.011151,3.379903,0.456208,0.01453,0.788640,0.054384,7,83,24
4,5,0.000000,652.678,0.0,0.00000,16,0.000000,0.0,223.637,454.636,...,0.165097,0.011151,3.800616,0.642546,0.01453,0.835205,0.054384,7,149,32


In [19]:
# Read in patient alias
patient_alias = pd.read_excel('../data/patient_alias_from_2025.xlsx', nrows= 40)
patient_alias_dict = dict(zip(patient_alias['PID'].astype(str), patient_alias['alias']))
patient_alias_dict

control_alias = pd.read_csv('../data/healthy_control.csv')
control_alias['key'] = control_alias['key'].str.replace('Healthy_BMA_67y_Male', 'Healthy_BMA_67Y_Male')

control_alias_dict = dict(zip(control_alias['key'].str.replace('_T_Cell_Panel.fcs', ''), control_alias['value'].str.replace('.fcs', '')))

In [20]:
cd8_t['FileName'].unique()

array(['61201_001_C1_D8_T_Cell_Panel', '61201_001_C7_D1_T_Cell_Panel',
       '61201_001_C7_D22_T_Cell_Panel', '61201_001_SPD_T_Cell_Panel',
       '61201_002_C1_D1_T_Cell_Panel', '61201_002_C1_D8_T_Cell_Panel',
       '61201_002_C7_D1_T_Cell_Panel', '61201_003_C1_D1_T_Cell_Panel',
       '61201_003_C1_D8_T_Cell_Panel', '61201_003_C7_D1_T_Cell_Panel',
       '61201_003_C7_D22_T_Cell_Panel', '61213_005_C1_D1_T_Cell_Panel',
       '61213_005_C1_D8_T_Cell_Panel', '61213_005_C12_D29_T_Cell_Panel',
       '61213_005_C7_D22_T_Cell_Panel', '61213_006_C1_D1_T_Cell_Panel',
       '61213_006_C1_D8_T_Cell_Panel', '61213_006_C12_D29_T_Cell_Panel',
       '61213_006_C7_D22_T_Cell_Panel', '61213_011_C1_D1_T_Cell_Panel',
       '61213_011_C1_D8_T_Cell_Panel', '61213_011_C12_D29_T_Cell_Panel',
       '61213_011_C7_D1_T_Cell_Panel', '61213_011_C7_D22_T_Cell_Panel',
       '61214_001_C1_D1_T_Cell_Panel', '61214_003_C1_D1_T_Cell_Panel',
       '61214_003_C1_D8_T_Cell_Panel', '61214_007_C12_D29_T_Cell_Pan

In [29]:
# Rename the filenames
filename_dict = {}
for i in cd8_t['FileName'].unique():
    if i.startswith('6'):
        x = i.split('_')[0] + '_' + i.split('_')[1]
        x = x.replace('_', '')
        z = '_'.join(i.split('_')[2:])
        filename_dict[i] = patient_alias_dict[x] + '_' + z
    elif i.startswith('Healthy'):
        #x = i.split('_')[0] + '_' + i.split('_')[1] + '_' + i.split('_')[2]
        x = i.replace('_T_Cell_Panel', '')
        filename_dict[i] = control_alias_dict[x] + '_T_Cell_Panel'

cd8_t['FileName'] = cd8_t['FileName'].map(filename_dict)

In [31]:
filename_dict

{'61201_001_C1_D8_T_Cell_Panel': 'P08_C1_D8_T_Cell_Panel',
 '61201_001_C7_D1_T_Cell_Panel': 'P08_C7_D1_T_Cell_Panel',
 '61201_001_C7_D22_T_Cell_Panel': 'P08_C7_D22_T_Cell_Panel',
 '61201_001_SPD_T_Cell_Panel': 'P08_SPD_T_Cell_Panel',
 '61201_002_C1_D1_T_Cell_Panel': 'P24_C1_D1_T_Cell_Panel',
 '61201_002_C1_D8_T_Cell_Panel': 'P24_C1_D8_T_Cell_Panel',
 '61201_002_C7_D1_T_Cell_Panel': 'P24_C7_D1_T_Cell_Panel',
 '61201_003_C1_D1_T_Cell_Panel': 'P19_C1_D1_T_Cell_Panel',
 '61201_003_C1_D8_T_Cell_Panel': 'P19_C1_D8_T_Cell_Panel',
 '61201_003_C7_D1_T_Cell_Panel': 'P19_C7_D1_T_Cell_Panel',
 '61201_003_C7_D22_T_Cell_Panel': 'P19_C7_D22_T_Cell_Panel',
 '61213_005_C1_D1_T_Cell_Panel': 'P12_C1_D1_T_Cell_Panel',
 '61213_005_C1_D8_T_Cell_Panel': 'P12_C1_D8_T_Cell_Panel',
 '61213_005_C12_D29_T_Cell_Panel': 'P12_C12_D29_T_Cell_Panel',
 '61213_005_C7_D22_T_Cell_Panel': 'P12_C7_D22_T_Cell_Panel',
 '61213_006_C1_D1_T_Cell_Panel': 'P04_C1_D1_T_Cell_Panel',
 '61213_006_C1_D8_T_Cell_Panel': 'P04_C1_D8_T_Cell

In [33]:
# Rename the filenames
patient_id_dict = {}
for i in cd8_t['Patient_ID'].unique():
    if i.startswith('6'):
        patient_id_dict[i] = patient_alias_dict[i.replace('_', '').replace('612500042', '61250004')]
    elif i.startswith('Healthy'):
        #x = i.split('_')[0] + '_' + i.split('_')[1] + '_' + i.split('_')[2]
        x = i.replace('_T_Cell_Panel', '')
        patient_id_dict[i] = control_alias_dict[x]

cd8_t['Patient_ID'] = cd8_t['Patient_ID'].map(patient_id_dict)

In [ ]:
cd8_t['FileName'].unique()

In [37]:
cd8_t.to_csv('../data/all_cd8_t_cells_patient_updated_patient_updated.csv')

In [35]:
cd8_t['Patient_ID']

0                 P08
1                 P08
2                 P08
3                 P08
4                 P08
              ...    
2343091    Control_11
2343092    Control_11
2343093    Control_11
2343094    Control_11
2343095    Control_11
Name: Patient_ID, Length: 2343096, dtype: object

In [39]:
cd8_t

,Unnamed: 0,209Bi_CD45,Center,164Dy,166Er_CD34,Event_length,157Gd,113In_CD45,191Ir_DNA1,193Ir_DNA2,...,152Sm_CD159c_asinh_coarseAlign_fineAlign,169Tm_CD25_asinh_coarseAlign_fineAlign,171Yb_Granzyme_B_asinh_coarseAlign_fineAlign,172Yb_CD38_asinh_coarseAlign_fineAlign,173Yb_CD14_asinh_coarseAlign_fineAlign,174Yb_HLA-DR_asinh_coarseAlign_fineAlign,176Yb_CD56_asinh_coarseAlign_fineAlign,Alignment_MC_fineAlign,FlowSOM_cluster,FlowSOM_metacluster
0,1,0.000000,703.693,0.0,0.000000,17,0.000000,0.0,218.3070,384.787,...,0.056854,0.011151,5.832982,2.280988,0.014530,0.647175,0.054384,7,70,22
1,2,0.050212,791.971,0.0,0.000000,20,0.000000,0.0,156.8950,337.880,...,0.006105,0.011151,3.614947,1.015876,0.014530,0.073580,0.054384,7,90,6
2,3,0.000000,0.000,0.0,7.968260,14,0.000000,0.0,139.2010,343.646,...,0.006105,0.011151,2.196237,2.859861,0.569240,0.437788,0.054384,7,194,39
3,4,0.000000,794.320,0.0,0.000000,17,0.927911,0.0,149.2950,344.580,...,0.006105,0.011151,3.379903,0.456208,0.014530,0.788640,0.054384,7,83,24
4,5,0.000000,652.678,0.0,0.000000,16,0.000000,0.0,223.6370,454.636,...,0.165097,0.011151,3.800616,0.642546,0.014530,0.835205,0.054384,7,149,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2343091,2343092,481.402000,586.396,0.0,4.785110,15,1.813500,0.0,95.8718,198.037,...,0.000441,0.007053,8.302777,2.284464,0.107306,0.373201,2.660276,7,32,3
2343092,2343093,457.100000,563.758,0.0,8.869000,14,0.244591,0.0,92.4527,288.428,...,0.000441,0.007053,7.215161,1.246797,0.010563,1.129578,3.033520,7,34,3
2343093,2343094,567.783000,623.361,0.0,0.000000,15,0.000000,0.0,107.8060,259.515,...,0.000441,0.007053,7.627989,1.586866,0.301672,1.210380,5.389370,7,20,3
2343094,2343095,425.390000,527.919,0.0,0.000000,13,0.000000,0.0,142.5490,226.593,...,0.000441,0.007053,4.340305,0.732783,0.010563,0.094293,3.992789,7,18,2
